# 03 · PyTorch CNN 影像分類

**對應教學 App**：🖼️ CNN 卷積神經網路

**你會做到**：
1. 用 sklearn 內建的手寫數字資料（**完全離線，不用下載**）
2. 建一個 CNN，跑完整訓練迴圈
3. 比較 CNN vs 全連接網路的參數量和表現
4. **把第一層學到的濾波器畫出來**——你會看到它自己學出了邊緣偵測器
5. 找出模型答錯的樣本，看它錯在哪

---

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

matplotlib.rcParams["font.sans-serif"] = ["Microsoft JhengHei", "Microsoft YaHei", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("使用裝置：", device)
print("（沒有 GPU 也完全跑得動，這個資料集很小）")

## 1. 載入資料

`load_digits` 是 8×8 的手寫數字（0～9），共 1797 張。
比 MNIST 小很多，好處是**幾秒鐘就訓練完**，適合學習。

In [ ]:
digits = load_digits()
X = digits.images.astype(np.float32) / 16.0        # 原始值 0~16，縮放到 0~1
y = digits.target.astype(np.int64)

print("圖片形狀:", X.shape, "  (張數, 高, 寬)")
print("標籤形狀:", y.shape)
print("類別:", np.unique(y))

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for ax, img, label in zip(axes.ravel(), X, y):
    ax.imshow(img, cmap="gray")
    ax.set_title(f"標籤 {label}")
    ax.axis("off")
plt.suptitle("原始資料長這樣（8×8 灰階）")
plt.tight_layout(); plt.show()

In [ ]:
# 切訓練 / 測試，並轉成 PyTorch 的 tensor
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# CNN 的輸入需要 (批次, 通道, 高, 寬) 四個維度 —— 灰階所以通道 = 1
Xtr = torch.tensor(X_train).unsqueeze(1).to(device)
Xte = torch.tensor(X_test).unsqueeze(1).to(device)
ytr = torch.tensor(y_train).to(device)
yte = torch.tensor(y_test).to(device)

print("訓練集:", Xtr.shape)
print("測試集:", Xte.shape)

## 2. 定義 CNN

架構：
```
輸入 (1, 8, 8)
  → Conv 3×3, 16 個濾波器 + BatchNorm + ReLU     (16, 8, 8)
  → Conv 3×3, 32 個濾波器 + BatchNorm + ReLU     (32, 8, 8)
  → MaxPool 2×2                                  (32, 4, 4)
  → Flatten                                      (512,)
  → Dropout + Linear(512 → 64) + ReLU
  → Linear(64 → 10)
輸出：10 個類別的分數
```

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),   # padding=1 讓尺寸不變
            nn.BatchNorm2d(16),
            nn.ReLU(),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),

            nn.MaxPool2d(2),                              # 8×8 → 4×4
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(32 * 4 * 4, 64), nn.ReLU(),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


cnn = SmallCNN().to(device)
print(cnn)
print()
print("參數總數:", sum(p.numel() for p in cnn.parameters()))

## 3. 訓練迴圈

**這四行的順序要背起來**（面試會叫你默寫）：
```python
optimizer.zero_grad()   # ① 清掉上一輪的梯度
loss.backward()         # ② 反向傳播算梯度
optimizer.step()        # ③ 更新權重
```

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

train_loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=64, shuffle=True)


def train_model(model, epochs=30, lr=1e-3):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    hist = {"train_loss": [], "test_acc": []}

    for ep in range(epochs):
        model.train()                      # 切到訓練模式（BatchNorm / Dropout 會生效）
        total = 0.0
        for xb, yb in train_loader:
            optimizer.zero_grad()          # ①
            loss = criterion(model(xb), yb)
            loss.backward()                # ②
            optimizer.step()               # ③
            total += loss.item() * len(xb)

        model.eval()                       # 切到推論模式（很重要，不然 BatchNorm 會用錯統計量）
        with torch.no_grad():
            acc = (model(Xte).argmax(1) == yte).float().mean().item()

        hist["train_loss"].append(total / len(Xtr))
        hist["test_acc"].append(acc)

        if ep % 5 == 0 or ep == epochs - 1:
            print(f"epoch {ep:3d}　loss {hist['train_loss'][-1]:.4f}　測試正確率 {acc:.4f}")

    return hist


hist_cnn = train_model(cnn)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(hist_cnn["train_loss"], color="#dc2626", lw=2)
axes[0].set_title("訓練損失"); axes[0].set_xlabel("epoch"); axes[0].grid(alpha=.3)
axes[1].plot(hist_cnn["test_acc"], color="#16a34a", lw=2)
axes[1].set_title("測試正確率"); axes[1].set_xlabel("epoch"); axes[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

## 4. 對照組：一樣的任務，用全連接網路

In [ ]:
class SimpleMLP(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),                       # 8×8 攤平成 64 —— 空間結構就此消失
            nn.Linear(64, 128), nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        return self.net(x)


mlp = SimpleMLP().to(device)
hist_mlp = train_model(mlp)

print()
print("=" * 46)
print(f"{'模型':<10}{'參數量':>12}{'測試正確率':>14}")
print("-" * 46)
print(f"{'CNN':<10}{sum(p.numel() for p in cnn.parameters()):>12,}{hist_cnn['test_acc'][-1]:>14.4f}")
print(f"{'MLP':<10}{sum(p.numel() for p in mlp.parameters()):>12,}{hist_mlp['test_acc'][-1]:>14.4f}")
print("=" * 46)
print()
print("💡 8×8 的小圖差距不明顯。圖片越大（224×224），CNN 的優勢會呈倍數放大——")
print("   因為 MLP 的參數量隨圖片面積線性成長，CNN 完全不受影響。")

## 5. 【最有趣的部分】把 CNN 學到的濾波器畫出來

第一層有 16 個 3×3 的濾波器。**沒有人告訴它們要長什麼樣**，
這些數字完全是訓練出來的。看看它們自己學成了什麼。

In [ ]:
filters = cnn.features[0].weight.detach().cpu().numpy()   # (16, 1, 3, 3)

fig, axes = plt.subplots(2, 8, figsize=(15, 4))
for i, ax in enumerate(axes.ravel()):
    f = filters[i, 0]
    ax.imshow(f, cmap="RdBu_r", vmin=-abs(filters).max(), vmax=abs(filters).max())
    ax.set_title(f"#{i}", fontsize=9)
    ax.axis("off")
plt.suptitle("第一層自己學到的 16 個 3×3 濾波器", fontsize=14)
plt.tight_layout(); plt.show()

print("👉 紅藍相間的方向性圖案 = 邊緣偵測器。")
print("   拿它們去對照教學 App「CNN」那頁的 Sobel 濾波器，長得很像。")
print("   模型自己發現了「偵測邊緣」是辨識數字的好方法。")

In [ ]:
# 看一張圖經過每個濾波器之後變成什麼樣（特徵圖 feature maps）
sample_idx = 0
sample = Xte[sample_idx:sample_idx + 1]

cnn.eval()
with torch.no_grad():
    fmaps = cnn.features[0](sample).cpu().numpy()[0]      # (16, 8, 8)

fig, axes = plt.subplots(2, 9, figsize=(17, 4.2))
axes[0, 0].imshow(sample.cpu().numpy()[0, 0], cmap="gray")
axes[0, 0].set_title(f"原圖（{yte[sample_idx].item()}）", fontsize=10)
axes[0, 0].axis("off")
axes[1, 0].axis("off")

for i in range(16):
    ax = axes[i // 8, i % 8 + 1]
    ax.imshow(fmaps[i], cmap="viridis")
    ax.set_title(f"濾波器 #{i}", fontsize=8)
    ax.axis("off")

plt.suptitle("同一張圖經過 16 個濾波器後的「特徵圖」——每個濾波器抓到不同的東西")
plt.tight_layout(); plt.show()

## 6. 錯在哪裡：看模型答錯的樣本

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

cnn.eval()
with torch.no_grad():
    logits = cnn(Xte)
    pred = logits.argmax(1).cpu().numpy()
    prob = F.softmax(logits, dim=1).max(1).values.cpu().numpy()

true = yte.cpu().numpy()
print(classification_report(true, pred, digits=3))

In [ ]:
cm = confusion_matrix(true, pred)

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm, cmap="Blues")
for i in range(10):
    for j in range(10):
        if cm[i, j] > 0:
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=9,
                    color="white" if cm[i, j] > cm.max() / 2 else "black")
ax.set_xlabel("預測"); ax.set_ylabel("實際"); ax.set_title("混淆矩陣")
ax.set_xticks(range(10)); ax.set_yticks(range(10))
plt.colorbar(im); plt.tight_layout(); plt.show()

print("👉 對角線以外的數字就是錯誤。看看哪兩個數字最容易被搞混。")

In [ ]:
wrong = np.where(pred != true)[0]
print(f"總共答錯 {len(wrong)} 張（共 {len(true)} 張）")

if len(wrong) > 0:
    n = min(10, len(wrong))
    fig, axes = plt.subplots(1, n, figsize=(2 * n, 2.6))
    axes = np.atleast_1d(axes)
    for ax, idx in zip(axes, wrong[:n]):
        ax.imshow(Xte[idx].cpu().numpy()[0], cmap="gray")
        ax.set_title(f"實際 {true[idx]}\n猜 {pred[idx]} ({prob[idx]:.0%})", fontsize=9)
        ax.axis("off")
    plt.suptitle("模型答錯的樣本")
    plt.tight_layout(); plt.show()

    print()
    print("💡 實務技巧：一定要親眼看錯誤樣本。")
    print("   常常會發現是「標籤本身標錯了」或「圖片本身人也看不出來」，")
    print("   這種情況再怎麼調模型都沒用。")

## 7. 算一次輸出尺寸（面試會考）

公式：`輸出邊長 = floor((輸入 + 2×padding − kernel) / stride) + 1`

In [ ]:
def conv_out(size, kernel, stride=1, padding=0):
    return (size + 2 * padding - kernel) // stride + 1

print("輸入 8×8")
print(f"→ Conv 3×3, padding=1, stride=1 → {conv_out(8, 3, 1, 1)}×{conv_out(8, 3, 1, 1)}  (尺寸不變)")
print(f"→ Conv 3×3, padding=0, stride=1 → {conv_out(8, 3, 1, 0)}×{conv_out(8, 3, 1, 0)}  (縮小 2)")
print(f"→ Conv 3×3, padding=1, stride=2 → {conv_out(8, 3, 2, 1)}×{conv_out(8, 3, 2, 1)}  (砍半)")
print(f"→ MaxPool 2×2                   → {conv_out(8, 2, 2, 0)}×{conv_out(8, 2, 2, 0)}  (砍半)")
print()
print("👉 用程式驗證一次，比死背公式牢固得多。")

# 用實際的 tensor 驗證
x = torch.randn(1, 1, 8, 8)
print()
print("實測：", x.shape, "→", nn.Conv2d(1, 4, 3, padding=1)(x).shape)

---

## 🎯 動手改改看

1. **把 `padding=1` 改成 `padding=0`**，重跑。
   → 會報錯，因為尺寸變了、`Linear` 的輸入維度對不上。
   自己算出新的維度並修正 —— **這是最好的練習**。

2. **拿掉 BatchNorm**，看訓練曲線變得多不穩。

3. **把 Dropout 從 0.3 改成 0.8**。
   → 訓練會變慢很多，因為每次都關掉 80% 的神經元。

4. **加資料增強**：訓練前隨機把圖片旋轉幾度
   （`torchvision.transforms.RandomRotation`），看正確率會不會提升。

5. **換成真的 MNIST**（28×28，7 萬張）：
   ```python
   from torchvision import datasets, transforms
   train = datasets.MNIST("./data", train=True, download=True,
                          transform=transforms.ToTensor())
   ```
   ⚠️ 這行需要網路連線下載。

## 📝 下一步

`04_Attention手刻實作.ipynb`